# Check 04 — Tenant and Limits

**Category:** Module smoke check (fast regression; companion to pytest, not a full tutorial).

**Purpose:** Prove `TenantQuotaManager` denies over-quota submissions (hard enforcement) and both in-memory and SQLite rate limiters enforce **fixed-window** per-tenant caps with tenant and limiter-id isolation.

**Prerequisites:**
- Python **3.12+** with project deps installed (`pip install -r requirements.txt` from repo root)
- Kernel: project **`.venv`** (see `notebooks/README.md`)
- Run cells **top to bottom** (bootstrap cell sets `sys.path` automatically)
- **No API key** required — deterministic, in-process only

**Related tutorial:** `tutorial_05_multi_turn_sessions.ipynb`

**Modules exercised:** `src/tenancy/quotas`, `src/tenancy/rate_limiter`

**PASS means:** Hard quota at boundary returns `TENANT_QUOTA_EXCEEDED`; soft quota returns `TENANT_QUOTA_SOFT_LIMIT` while still allowing; in-memory fixed-window limiter blocks the third request for tenant-a with exact remaining/retry counts; SQLite fixed-window limiter blocks the second request; other tenants/limiter ids remain allowed; final line `PASS: tenancy quota and fixed-window limiter checks`.

**Troubleshooting:** If SQLite limiter fails on permissions, confirm temp directory is writable. Quota uses `active_jobs` argument — must pass `2` to trigger denial when max is 2.

In [1]:
import pathlib
import sys

_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(_root))

import tempfile

from src.tenancy.quotas import TenantQuotaManager
from src.tenancy.rate_limiter import TenantRateLimiter, SQLiteTenantRateLimiter

In [2]:
quota = TenantQuotaManager(max_active_jobs_per_tenant=2, hard_enforcement=True)
assert quota.check_submission("tenant-a", active_jobs=0).allowed
assert quota.check_submission("tenant-a", active_jobs=1).allowed
denied = quota.check_submission("tenant-a", active_jobs=2)
assert denied.allowed is False
assert denied.reason_code == "TENANT_QUOTA_EXCEEDED"
assert "tenant-a" in denied.message

soft_quota = TenantQuotaManager(max_active_jobs_per_tenant=2, hard_enforcement=False)
soft = soft_quota.check_submission("tenant-soft", active_jobs=2)
assert soft.allowed is True
assert soft.reason_code == "TENANT_QUOTA_SOFT_LIMIT"

mem_limiter = TenantRateLimiter(max_requests=2, window_seconds=60)
a1, rem1 = mem_limiter.allow("tenant-a")
a2, rem2 = mem_limiter.allow("tenant-a")
a3, retry = mem_limiter.allow("tenant-a")
assert a1 is True and rem1 == 1
assert a2 is True and rem2 == 0
assert a3 is False and retry == 60

b1, b_rem1 = mem_limiter.allow("tenant-b")
assert b1 is True and b_rem1 == 1

unlimited = TenantRateLimiter(max_requests=0, window_seconds=60)
assert unlimited.allow("tenant-any") == (True, 0)

with tempfile.TemporaryDirectory() as tmp:
    db_path = str(pathlib.Path(tmp) / "limits.db")
    sqlite_limiter = SQLiteTenantRateLimiter(
        db_path=db_path, max_requests=1, window_seconds=60, limiter_id="turns"
    )
    ok1, rem_sql1 = sqlite_limiter.allow("tenant-b")
    ok2, retry2 = sqlite_limiter.allow("tenant-b")
    assert ok1 is True and rem_sql1 == 0
    assert ok2 is False and retry2 == 60

    ok_other_tenant, _ = sqlite_limiter.allow("tenant-c")
    assert ok_other_tenant is True

    ok_other_limiter, _ = sqlite_limiter.allow("tenant-b", limiter_id="other")
    assert ok_other_limiter is True

print("quota denial:", denied.reason_code, denied.message)
print("soft quota:", soft.reason_code, soft.allowed)
print(
    "memory limiter tenant-a:",
    [a1, a2, a3],
    "remaining/retry:",
    [rem1, rem2, retry],
)
print("memory limiter tenant-b isolated:", b1, b_rem1)
print("unlimited limiter:", unlimited.allow("tenant-any"))
print(
    "sqlite limiter tenant-b:",
    [ok1, ok2],
    "remaining/retry:",
    [rem_sql1, retry2],
)
print("sqlite tenant/limiter isolation: PASS")
print("PASS: tenancy quota and fixed-window limiter checks")

quota denial: TENANT_QUOTA_EXCEEDED Tenant 'tenant-a' exceeded max active jobs quota.
soft quota: TENANT_QUOTA_SOFT_LIMIT True
memory limiter tenant-a: [True, True, False] remaining/retry: [1, 0, 60]
memory limiter tenant-b isolated: True 1
unlimited limiter: (True, 0)
sqlite limiter tenant-b: [True, False] remaining/retry: [0, 60]
sqlite tenant/limiter isolation: PASS
PASS: tenancy quota and fixed-window limiter checks
